# PUMS revision runs: lesion-level joint CRC (pre-registration Amendment 3)

Runs the **frozen-model protocol** on a Kaggle GPU: download data from the official sources, train each
backbone once on 40% of the units, freeze it, predict the other 60%, then run all conformal calibration,
the clustered bootstrap, the ablations, and the aggregation. **Output:** `/kaggle/working/revision_results_<JOB>.zip`
(small; contains no images).

**Settings (right panel):** Accelerator = GPU (T4 x2 or P100) · Internet = On · Persistence = off.
Run with **Save Version → Save & Run All (Commit)**. Do not use interactive mode; a commit runs for up to 12 h unattended.

| Session | `JOB` | Approx. time |
|---|---|---|
| 1 | `"2d"` | Kvasir-SEG + FracAtlas, 2 backbones x 3 seeds, calibration + ablations: ~4-6 h |
| 2 | `"task08"`, `ARCHS=["unet-r34"]` | MSD Task08 liver tumours (patient-disjoint), 3 seeds: ~5-7 h |
| 3 | `"task08"`, `ARCHS=["unet-mitb2"]` | same, second backbone: ~6-8 h |

In [ ]:
# ======== CONFIG: edit before each session ========
JOB   = "task08"                              # "2d" or "task08"
ARCHS = ["unet-mitb2"]        # backbones for this session
SEEDS = [0, 1, 2]                         # model seeds (independent replicates)
RUN_SMOKE = True                          # 1-epoch, <=40-unit smoke test first (outputs deleted)

In [ ]:
import subprocess, sys, os, shutil, json, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "nibabel"], check=True)
import torch, segmentation_models_pytorch as smp
print("torch", torch.__version__, "| smp", smp.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), ("NO GPU: set Settings -> Accelerator -> 'GPU T4 x2' in the right panel, "
                                   "then Save Version -> Save & Run All (Commit) again")
os.system("df -h /tmp /kaggle/working | cat; nproc; free -g")

## Code (verbatim copies of the repository files)

In [ ]:
for d in ["backend/app/calibration", "backend/app/core", "backend/scripts/experiments", "docs/research/results_v2"]:
    os.makedirs(f"/kaggle/working/pums/{d}", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/pums/backend/app/__init__.py
# package marker

In [ ]:
%%writefile /kaggle/working/pums/backend/app/calibration/__init__.py
# package marker

In [ ]:
%%writefile /kaggle/working/pums/backend/app/core/__init__.py
# package marker

In [ ]:
%%writefile /kaggle/working/pums/backend/app/calibration/families.py
r"""Nested prediction-set families for segmentation risk control.

Every family here is a *fixed* score transform :math:`s = T(\hat p)` with values
in :math:`[0, 1]`, and its prediction sets are upper level sets

.. math:: S_\lambda = \{\, u : s(u) \ge 1 - \lambda \,\}, \qquad \lambda \in [0, 1].

Because :math:`T` does not depend on the label or on :math:`\lambda`, the sets are
nested in :math:`\lambda` for every image, which is all that Lemma 1 of the theory
memo needs for any set-monotone loss (pixel FNR, lesion miss, small-lesion miss)
to be non-increasing in :math:`\lambda`. Validity of CRC therefore holds for
every family; families differ only in *efficiency* — how large the set must be to
reach a given risk.

Families
--------
``threshold``          :math:`s=\hat p`. Standard CRC (Angelopoulos et al., 2024).
``dilation``           :math:`s = 1/(1+d(u,\hat M))`, :math:`\hat M=\{\hat p\ge\frac12\}`:
                       the level sets are :math:`\hat M\oplus B_r` — the morphological
                       prediction sets of Mossina & Friedrich (MICCAI 2025).
``lesion_equalized``   the contribution of this work, defined below.

Lesion-equalized scores
-----------------------
Let :math:`C_1,\dots,C_J` be the connected components of the *candidate* mask
:math:`\{\hat p \ge t_0\}` and :math:`P_j=\max_{u\in C_j}\hat p(u)` their peaks. Then

.. math::
   s_{\rho,t_0}(u) = \begin{cases}
       \hat p(u)\,/\,P_j^{\rho} & u \in C_j \\
       \hat p(u)               & \text{otherwise}
   \end{cases}\qquad \rho\in[0,1].

Properties (proved in ``docs/research/theory-memo-v2.md``):

* :math:`s\in[0,1]`, and every candidate pixel outranks every non-candidate
  pixel (inside, :math:`s \ge \hat p \ge t_0`; outside, :math:`s=\hat p<t_0`).
* :math:`\rho=0` recovers thresholding exactly, so the family strictly contains
  the standard CRC baseline.
* **Amplitude invariance at** :math:`\rho=1`: if a lesion's posterior is a scaled
  profile :math:`\hat p = a_k\,g` on its candidate component, then
  :math:`s = g/\max g`, independent of the confidence amplitude :math:`a_k`. A
  faint lesion is then hit at the *same* cutoff as a confident lesion of the same
  shape, so the global cutoff no longer has to fall to the faintest lesion's
  level — the mechanism by which pixel thresholding inflates sets everywhere to
  protect small, low-confidence lesions.

The pair :math:`(\rho, t_0)` must be chosen on data **disjoint** from the
calibration split (see :func:`app.calibration.joint.select_family`); choosing it
on the calibration split would void the guarantee.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy import ndimage

#: 8-connectivity (2-D) / 26-connectivity (3-D) structuring elements.
def connectivity(ndim: int) -> np.ndarray:
    return np.ones((3,) * ndim, dtype=bool)


@dataclass(frozen=True, slots=True)
class ScoreFamily:
    """A fixed, label-free score transform defining nested sets {s >= 1 - lambda}."""

    kind: str
    rho: float = 0.0
    t0: float = 0.5

    @property
    def label(self) -> str:
        if self.kind == "lesion_equalized":
            return f"lesion_equalized(rho={self.rho:g},t0={self.t0:g})"
        return self.kind

    def score(self, p: np.ndarray) -> np.ndarray:
        p = np.clip(np.asarray(p, dtype=np.float64), 0.0, 1.0)
        if self.kind == "threshold":
            return p
        if self.kind == "dilation":
            return dilation_score(p)
        if self.kind == "lesion_equalized":
            return lesion_equalized_score(p, rho=self.rho, t0=self.t0)
        raise ValueError(f"Unknown family '{self.kind}'")


def dilation_score(p: np.ndarray) -> np.ndarray:
    """1/(1+EDT to the argmax mask). Level sets are dilations of M_hat.

    If the argmax mask is empty the dilation family degenerates to the empty set
    for every radius (the 'feasibility floor' of theory memo §7). We seed it
    with the single highest-probability pixel instead so the family is never
    empty; this is disclosed and is the top-q seeding with q = 1 pixel.
    """
    seed = p >= 0.5
    if not seed.any():
        seed = p >= p.max()
    distance = ndimage.distance_transform_edt(~seed)
    return 1.0 / (1.0 + distance)


def lesion_equalized_score(p: np.ndarray, *, rho: float, t0: float) -> np.ndarray:
    """Peak-normalise each candidate component of {p >= t0} by P_j ** rho."""
    if not 0.0 <= rho <= 1.0:
        raise ValueError("rho must lie in [0, 1]")
    if not 0.0 < t0 <= 1.0:
        raise ValueError("t0 must lie in (0, 1]")
    s = p.copy()
    if rho == 0.0:
        return s
    candidate = p >= t0
    labels, count = ndimage.label(candidate, structure=connectivity(p.ndim))
    if count == 0:
        return s
    peaks = ndimage.maximum(p, labels, index=np.arange(1, count + 1))
    divisor = np.ones(count + 1, dtype=np.float64)
    divisor[1:] = np.power(np.maximum(peaks, t0), rho)
    inside = labels > 0
    s[inside] = np.minimum(p[inside] / divisor[labels[inside]], 1.0)
    return s


def default_family_grid() -> list[ScoreFamily]:
    """The tuning grid for lesion_equalized (rho = 0 is plain thresholding)."""
    grid = [ScoreFamily("threshold")]
    for t0 in (0.05, 0.1, 0.2, 0.3):
        for rho in (0.25, 0.5, 0.75, 1.0):
            grid.append(ScoreFamily("lesion_equalized", rho=rho, t0=t0))
    return grid

In [ ]:
%%writefile /kaggle/working/pums/backend/app/calibration/losses.py
r"""Structured, set-monotone segmentation losses on a fixed lambda grid.

For one calibration unit (an image or a volume) with ground truth :math:`Y` and
ground-truth connected components :math:`Y^1..Y^K`, and a score map :math:`s`
defining :math:`S_\lambda=\{s\ge 1-\lambda\}`:

.. math::
   L^{pix}(\lambda) &= 1 - |S_\lambda\cap Y| / |Y| \\
   L^{les}(\lambda) &= \tfrac1K\,\#\{k : |S_\lambda\cap Y^k| < \tau |Y^k|\} \\
   L^{small}(\lambda) &= \tfrac1{K_s}\,\#\{k \in \text{small} : |S_\lambda\cap Y^k| < \tau |Y^k|\}

All three are non-increasing in :math:`\lambda` for any nested family (theory memo,
Lemma 1). Units with :math:`Y=\emptyset` are excluded from all three; units with no
small lesion are excluded from :math:`L^{small}` only — each guarantee is therefore
*conditional on the structure being present*, and the reported :math:`n` is per loss.

Everything is reduced to integer counts on the grid once per unit (Lemma 2: a
component is hit at cutoff :math:`c` iff its :math:`\lceil\tau|Y^k|\rceil`-th largest
score is :math:`\ge c`), so re-calibrating on thousands of random splits costs a
few array sums rather than a pass over pixels.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy import ndimage

from app.calibration.families import connectivity

#: Lambda grid. 1001 points => 0.001 resolution; finer than the CRC finite-sample
#: slack 1/(n+1) for every calibration size used in this project.
GRID_SIZE = 1001
#: Default lesion hit fraction (theory memo §3, fixed before any result).
DEFAULT_TAU = 0.5


def lambda_grid(size: int = GRID_SIZE) -> np.ndarray:
    return np.linspace(0.0, 1.0, size)


@dataclass(slots=True)
class UnitStats:
    """Grid-reduced loss ingredients for one unit under one score family."""

    n_pixels: int
    n_gt: int
    #: [G] number of ground-truth pixels outside S_lambda.
    pix_miss: np.ndarray
    #: [G] |S_lambda|.
    set_size: np.ndarray
    #: [K] tau-order statistic of each GT component (Lemma 2).
    lesion_stat: np.ndarray
    #: [K] component sizes in pixels/voxels.
    lesion_size: np.ndarray

    @property
    def k(self) -> int:
        return int(self.lesion_stat.size)


def gt_components(mask: np.ndarray, min_size: int = 0) -> np.ndarray:
    """Label GT components with full connectivity, dropping those below min_size."""
    labels, count = ndimage.label(mask, structure=connectivity(mask.ndim))
    if count == 0 or min_size <= 0:
        return labels
    sizes = ndimage.sum(np.ones_like(labels), labels, index=np.arange(1, count + 1))
    keep = np.flatnonzero(sizes >= min_size) + 1
    filtered = np.where(np.isin(labels, keep), labels, 0)
    return ndimage.label(filtered > 0, structure=connectivity(mask.ndim))[0]


def unit_stats(
    score: np.ndarray,
    labels: np.ndarray,
    *,
    tau: float = DEFAULT_TAU,
    lambdas: np.ndarray | None = None,
) -> UnitStats:
    """Reduce one unit's score map + labelled GT to grid counts."""
    lambdas = lambda_grid() if lambdas is None else lambdas
    cutoffs = 1.0 - lambdas
    flat = score.ravel()
    lab = labels.ravel()
    truth = lab > 0

    all_sorted = np.sort(flat)
    set_size = flat.size - np.searchsorted(all_sorted, cutoffs, side="left")
    gt_sorted = np.sort(flat[truth])
    pix_miss = np.searchsorted(gt_sorted, cutoffs, side="left")

    k = int(lab.max()) if truth.any() else 0
    stats = np.empty(k)
    sizes = np.empty(k, dtype=np.int64)
    if k:
        order = np.argsort(lab[truth], kind="stable")
        comp_ids = lab[truth][order]
        comp_scores = flat[truth][order]
        bounds = np.searchsorted(comp_ids, np.arange(1, k + 2))
        for j in range(k):
            sc = np.sort(comp_scores[bounds[j] : bounds[j + 1]])[::-1]
            sizes[j] = sc.size
            stats[j] = sc[int(np.ceil(tau * sc.size)) - 1]
    return UnitStats(
        n_pixels=int(flat.size),
        n_gt=int(truth.sum()),
        pix_miss=pix_miss.astype(np.int64),
        set_size=set_size.astype(np.int64),
        lesion_stat=stats,
        lesion_size=sizes,
    )


@dataclass(slots=True)
class StackedStats:
    """UnitStats for many units, stacked for vectorised curve computation.

    Only units with non-empty GT are stacked; empty-GT units contribute set size
    only and are kept in ``empty_set_size``.
    """

    pix_rate: np.ndarray  # [N, G] per-unit pixel FNR
    les_rate: np.ndarray  # [N, G] per-unit lesion-miss rate
    small_rate: np.ndarray  # [N, G] per-unit small-lesion miss rate (nan if none)
    has_small: np.ndarray  # [N] bool
    set_size: np.ndarray  # [N, G]
    n_gt: np.ndarray  # [N]
    n_pixels: np.ndarray  # [N]
    multi: np.ndarray  # [N] bool, K >= 2

    def __len__(self) -> int:
        return int(self.n_gt.size)

    def subset(self, idx: np.ndarray) -> "StackedStats":
        return StackedStats(*(getattr(self, f)[idx] for f in self.__slots__))


def stack(units: list[UnitStats], *, small_max_size: float, lambdas: np.ndarray | None = None) -> StackedStats:
    """Stack non-empty units; ``small_max_size`` is the small-lesion size cut (exclusive)."""
    lambdas = lambda_grid() if lambdas is None else lambdas
    cutoffs = 1.0 - lambdas
    kept = [u for u in units if u.n_gt > 0 and u.k > 0]
    n, g = len(kept), lambdas.size
    pix = np.empty((n, g), dtype=np.float32)
    les = np.empty((n, g), dtype=np.float32)
    small = np.full((n, g), np.nan, dtype=np.float32)
    has_small = np.zeros(n, dtype=bool)
    for i, u in enumerate(kept):
        pix[i] = u.pix_miss / u.n_gt
        st = np.sort(u.lesion_stat)
        les[i] = np.searchsorted(st, cutoffs, side="left") / st.size
        sm = np.sort(u.lesion_stat[u.lesion_size < small_max_size])
        if sm.size:
            has_small[i] = True
            small[i] = np.searchsorted(sm, cutoffs, side="left") / sm.size
    return StackedStats(
        pix_rate=pix,
        les_rate=les,
        small_rate=small,
        has_small=has_small,
        set_size=np.stack([u.set_size for u in kept]) if kept else np.empty((0, g)),
        n_gt=np.asarray([u.n_gt for u in kept]),
        n_pixels=np.asarray([u.n_pixels for u in kept]),
        multi=np.asarray([u.k >= 2 for u in kept]),
    )

In [ ]:
%%writefile /kaggle/working/pums/backend/scripts/experiments/kaggle_train.py
"""Frozen-model training for the revision runs, self-contained for Kaggle Notebooks.

Same protocol as train_frozen.py / train_frozen_msd.py (pre-registration
Amendment 3, P1). The dataset readers replicate app/datasets/loaders.py exactly
(Kvasir-SEG image/mask pairs; FracAtlas COCO polygons rasterised at native
resolution, lesion-free radiographs as training negatives; MSD Task08 tumours,
label 2, as 26-connected 3-D components), so the code runs without the rest of
the PUMS backend.

    python kaggle_train.py --dataset kvasir-seg --arch unet-r34 --seed 0 --data-root /tmp/pums_data

Output (identical format to train_frozen.py):
    <BACKEND_ROOT>/var/posteriors_frozen/<dataset>/<arch>_seed<m>.npz + .json      (2-D)
    <BACKEND_ROOT>/var/posteriors_frozen/<dataset>/<arch>_seed<m>/<case>.npz + .json (3-D)
"""

from __future__ import annotations

import argparse
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
from scipy import ndimage

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from app.calibration.families import connectivity  # noqa: E402
from app.calibration.losses import gt_components  # noqa: E402
from app.core.config import BACKEND_ROOT  # noqa: E402

OUT_ROOT = BACKEND_ROOT / "var" / "posteriors_frozen"
TRAIN_FRACTION = 0.4
VAL_FRACTION = 0.1
SMALL_QUANTILES = (25.0, 33.0, 50.0)
MIN_COMPONENT_PX = 10          # 2-D, native resolution (as train_backbone.py)
MIN_COMPONENT_VOX = 10         # 3-D Task08 (as train_backbone_msd.py)
HU_LO, HU_HI = -100.0, 250.0   # liver CT window (as train_backbone_msd.py)
NEG_SLICE_FRACTION = 0.1
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
ARCHS = ("unet-r34", "unet-mitb2", "fpn-effb3")


# ----------------------------------------------------------------- data (replicates app/datasets/loaders.py)
def _gray(path):
    from PIL import Image

    with Image.open(path) as h:
        return np.asarray(h.convert("L"), dtype=np.float64) / 255.0


def load_kvasir(root: Path):
    mask_dir = next(d for d in sorted(root.rglob("*")) if d.is_dir() and d.name.lower() in {"masks", "mask", "annotations"})
    image_dir = next(d for d in sorted(root.rglob("*")) if d.is_dir() and d.name.lower() in {"images", "image", "img"})
    from PIL import Image

    for mask_path in sorted(mask_dir.iterdir()):
        if mask_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        cands = [image_dir / mask_path.name, *(image_dir / f"{mask_path.stem}{e}" for e in IMAGE_EXTENSIONS)]
        ip = next((c for c in cands if c.exists()), None)
        if ip is None:
            continue
        image = _gray(ip)
        with Image.open(mask_path) as h:
            mask = np.asarray(h.convert("L")) > 127
        if mask.shape != image.shape or not mask.any():
            continue
        yield image, mask, mask_path.stem


def load_fracatlas(root: Path):
    from PIL import Image, ImageDraw

    coco_path = next(root.rglob("COCO_fracture_masks.json"))
    coco = json.loads(coco_path.read_text(encoding="utf-8"))
    by_image: dict[int, list] = {}
    for ann in coco["annotations"]:
        by_image.setdefault(ann["image_id"], []).append(ann)
    image_dir = next(d for d in root.rglob("Fractured") if d.is_dir())
    for info in sorted(coco["images"], key=lambda r: r["file_name"]):
        path = image_dir / info["file_name"]
        anns = by_image.get(info["id"], [])
        if not path.exists() or not anns:
            continue
        image = _gray(path)
        canvas = Image.new("L", (image.shape[1], image.shape[0]), 0)
        draw = ImageDraw.Draw(canvas)
        for ann in anns:
            seg = ann["segmentation"]
            for poly in (json.loads(seg) if isinstance(seg, str) else seg):
                if len(poly) >= 6:
                    draw.polygon(list(map(float, poly)), outline=1, fill=1)
        mask = np.asarray(canvas) > 0
        if mask.any():
            yield image, mask, Path(info["file_name"]).stem


def fracatlas_negatives(root: Path):
    neg_dir = next(d for d in root.rglob("Non_fractured") if d.is_dir())
    skipped = 0
    for p in sorted(neg_dir.iterdir()):
        if p.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                yield _gray(p)
            except OSError as exc:  # a corrupt/truncated file must not stop the run; it is counted
                skipped += 1
                print(f"  skipped unreadable negative {p.name}: {exc}", flush=True)
    if skipped:
        print(f"  {skipped} lesion-free image(s) skipped", flush=True)


def letterbox(arr, size, *, nearest):
    import cv2

    h, w = arr.shape
    side = max(h, w)
    canvas = np.zeros((side, side), dtype=arr.dtype)
    canvas[:h, :w] = arr
    valid = np.zeros((side, side), dtype=np.uint8)
    valid[:h, :w] = 1
    out = cv2.resize(canvas, (size, size), interpolation=cv2.INTER_NEAREST if nearest else cv2.INTER_AREA)
    return out, cv2.resize(valid, (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)


def prepare_2d(dataset, data_root: Path, size, max_units=None):
    # Streamed: each image is resized as soon as it is read (FracAtlas radiographs are
    # ~1760x2140; holding the 3366 lesion-free ones at native resolution would need ~100 GB).
    root = data_root / dataset
    pos = load_kvasir(root) if dataset == "kvasir-seg" else load_fracatlas(root)
    neg = fracatlas_negatives(root) if dataset == "fracatlas" else iter(())
    import itertools

    pos, neg = itertools.islice(pos, max_units), itertools.islice(neg, max_units)
    images, labels, valids, ids = [], [], [], []
    for image, mask, cid in pos:
        lab = gt_components(mask, MIN_COMPONENT_PX).astype(np.uint16)
        img, v = letterbox(image.astype(np.float32), size, nearest=False)
        lab_r, _ = letterbox(lab, size, nearest=True)
        images.append((np.clip(img, 0, 1) * 255).astype(np.uint8))
        labels.append(lab_r.astype(np.uint8))
        valids.append(v)
        ids.append(cid)
    x_neg = [(np.clip(letterbox(im.astype(np.float32), size, nearest=False)[0], 0, 1) * 255).astype(np.uint8)
             for im in neg]
    return (np.stack(images), np.stack(labels), np.stack(valids), ids,
            np.stack(x_neg) if x_neg else np.empty((0, size, size), np.uint8))


def load_volume(image_path, label_path, size):
    import cv2
    import nibabel as nib

    img = np.asarray(nib.load(str(image_path)).dataobj, dtype=np.float32)
    lab = np.asarray(nib.load(str(label_path)).dataobj)
    comps = gt_components(lab == 2, MIN_COMPONENT_VOX)
    assert comps.max() < 256, "more than 255 tumours in one volume"
    comps = comps.astype(np.uint8)
    img = np.clip((img - HU_LO) / (HU_HI - HU_LO), 0, 1)
    d = img.shape[2]
    x = np.empty((d, size, size), np.uint8)
    y = np.empty((d, size, size), np.uint8)
    for z in range(d):
        x[z] = (cv2.resize(img[:, :, z], (size, size), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
        y[z] = cv2.resize(comps[:, :, z], (size, size), interpolation=cv2.INTER_NEAREST)
    return x, y


# ----------------------------------------------------------------- model (as train_frozen.py)
def build(arch):
    import segmentation_models_pytorch as smp

    if arch == "unet-r34":
        return smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1)
    if arch == "unet-mitb2":
        return smp.Unet("mit_b2", encoder_weights="imagenet", in_channels=3, classes=1)
    if arch == "fpn-effb3":
        return smp.FPN("efficientnet-b3", encoder_weights="imagenet", in_channels=3, classes=1)
    raise ValueError(arch)


def to_tensor(xb, device, mean, std):
    import torch

    xt = torch.from_numpy(np.ascontiguousarray(xb, dtype=np.float32)).to(device).unsqueeze(1).repeat(1, 3, 1, 1)
    return (xt - mean) / std


def predict(model, x, mean, std, device="cuda", batch=8):
    import torch

    model.eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for b in range(0, len(x), batch):
            xt = to_tensor(x[b:b + batch].astype(np.float32) / 255.0, device, mean, std)
            p = torch.sigmoid(model(xt)) + torch.sigmoid(model(xt.flip(-1))).flip(-1)
            out.append((p / 2).squeeze(1).float().cpu().numpy())
    return np.concatenate(out) if out else np.empty((0,) + x.shape[1:], np.float32)


def mean_dice(prob, labels, valid):
    d = []
    for p, y, v in zip(prob, labels, valid):
        a, b = (p >= 0.5) & v, y > 0
        d.append(2 * (a & b).sum() / max(a.sum() + b.sum(), 1))
    return float(np.mean(d))


def train(arch, x_tr, y_tr, x_val, y_val, v_val, x_neg, *, epochs, seed, batch, device="cuda"):
    import torch
    import torch.nn.functional as F

    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = True
    rng = np.random.default_rng(seed)
    model = build(arch).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    steps = int(np.ceil(len(x_tr) / batch))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epochs * steps)
    scaler = torch.amp.GradScaler("cuda")
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    best, best_state, curve = -1.0, None, []
    for ep in range(epochs):
        model.train()
        order = rng.permutation(len(x_tr))
        losses = []
        for b in range(steps):
            idx = order[b * batch:(b + 1) * batch]
            xb, yb = x_tr[idx], (y_tr[idx] > 0).astype(np.float32)
            if len(x_neg):
                nidx = rng.integers(0, len(x_neg), max(1, len(idx) // 2))
                xb = np.concatenate([xb, x_neg[nidx]])
                yb = np.concatenate([yb, np.zeros((len(nidx),) + yb.shape[1:], np.float32)])
            if rng.random() < 0.5:
                xb, yb = xb[:, :, ::-1], yb[:, :, ::-1]
            if rng.random() < 0.5:
                xb, yb = xb[:, ::-1], yb[:, ::-1]
            xb = (xb.astype(np.float32) / 255.0) ** rng.uniform(0.7, 1.4)
            xt = to_tensor(xb, device, mean, std)
            yt = torch.from_numpy(np.ascontiguousarray(yb)).to(device).unsqueeze(1)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(xt)
                prob = torch.sigmoid(logits)
                inter = (prob * yt).sum((2, 3))
                dice = 1 - (2 * inter + 1) / (prob.sum((2, 3)) + yt.sum((2, 3)) + 1)
                loss = F.binary_cross_entropy_with_logits(logits, yt) + dice.mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(float(loss))
        vd = mean_dice(predict(model, x_val, mean, std, device), y_val, v_val)
        curve.append(dict(epoch=ep + 1, loss=float(np.mean(losses)), val_dice=vd))
        if vd > best:
            best = vd
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        print(f"    epoch {ep + 1:3d}  loss {np.mean(losses):.4f}  val Dice {vd:.3f}{'  *' if vd == best else ''}",
              flush=True)
    model.load_state_dict(best_state)
    return model, mean, std, curve, best


def lesion_metrics(prob, labels, valid, tau=0.5):
    dice, iou, rec_hits, rec_n, prec_hits, prec_n, fp = [], [], 0, 0, 0, 0, []
    for p, y, v in zip(prob, labels, valid):
        pred, gt = (p >= 0.5) & v, y > 0
        inter, union = (pred & gt).sum(), (pred | gt).sum()
        dice.append(2 * inter / max(pred.sum() + gt.sum(), 1))
        iou.append(inter / max(union, 1))
        for k in range(1, int(y.max()) + 1):
            comp = y == k
            if comp.any():
                rec_n += 1
                rec_hits += int((pred & comp).sum() >= tau * comp.sum())
        lab, n = ndimage.label(pred, structure=connectivity(pred.ndim))
        prec_n += n
        hit = np.unique(lab[gt & (lab > 0)])
        prec_hits += hit.size
        fp.append(n - hit.size)
    return dict(dice=float(np.mean(dice)), iou=float(np.mean(iou)),
                lesion_recall=rec_hits / max(rec_n, 1), lesion_precision=prec_hits / max(prec_n, 1),
                fp_components_per_unit=float(np.mean(fp)), n_lesions=rec_n, n_units=len(prob))


def small_cuts(label_arrays):
    sizes = [np.bincount(y.ravel())[1:] for y in label_arrays if y.max()]
    sizes = np.concatenate(sizes) if sizes else np.zeros(0)
    sizes = sizes[sizes > 0]
    return {f"q{q:g}": float(np.percentile(sizes, q)) for q in SMALL_QUANTILES}


def split(n, seed):
    rng = np.random.default_rng(5000 + seed)
    perm = rng.permutation(n)
    n_train = int(round(TRAIN_FRACTION * n))
    tr_all, pool = np.sort(perm[:n_train]), np.sort(perm[n_train:])
    vperm = rng.permutation(tr_all)
    n_val = max(1, int(round(VAL_FRACTION * len(tr_all))))
    return rng, np.sort(vperm[n_val:]), np.sort(vperm[:n_val]), tr_all, pool


def env_info():
    import segmentation_models_pytorch as smp
    import torch

    return dict(torch=torch.__version__, smp=smp.__version__, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0), python=platform.python_version(), platform="kaggle")


def run_2d(args):
    t0 = time.time()
    x, y, valid, ids, x_neg = prepare_2d(args.dataset, Path(args.data_root), args.size, args.max_units)
    rng, tr, val, tr_all, pool = split(len(x), args.seed)
    neg_tr = x_neg[rng.permutation(len(x_neg))[:int(round(TRAIN_FRACTION * len(x_neg)))]] if len(x_neg) else x_neg
    print(f"{args.dataset} {args.arch} seed {args.seed}: {len(x)} units -> train {len(tr)} / val {len(val)} / "
          f"pool {len(pool)}; {len(neg_tr)} lesion-free training images; prep {time.time() - t0:.0f}s", flush=True)
    model, mean, std, curve, best_val = train(args.arch, x[tr], y[tr], x[val], y[val], valid[val], neg_tr,
                                              epochs=args.epochs, seed=args.seed, batch=args.batch)
    prob = predict(model, x[pool], mean, std)
    metrics = lesion_metrics(prob, y[pool], valid[pool])
    cuts = small_cuts(y[tr_all])
    out_dir = args.out_root / args.dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{args.arch}_seed{args.seed}.npz"
    np.savez_compressed(path, prob=prob.astype(np.float16), labels=y[pool], valid=valid[pool], ids=np.asarray(ids)[pool])
    manifest = dict(protocol="P1 frozen model (pre-registration Amendment 3)", dataset=args.dataset, arch=args.arch,
                    seed=args.seed, epochs=args.epochs, size=args.size, batch=args.batch, n_units=len(x),
                    n_train=len(tr), n_val=len(val), n_pool=len(pool), n_negatives_train=len(neg_tr),
                    train_ids=[str(ids[i]) for i in tr], val_ids=[str(ids[i]) for i in val],
                    pool_ids=[str(ids[i]) for i in pool], best_val_dice=best_val, curve=curve, pool_metrics=metrics,
                    small_cuts_from_train=cuts, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                    seconds=round(time.time() - t0), env=env_info())
    path.with_suffix(".json").write_text(json.dumps(manifest, indent=2))
    print(f"  pool metrics {json.dumps(metrics)}\n  small cuts {cuts}\n  saved {path} ({manifest['seconds']}s)", flush=True)


def run_task08(args):
    t0 = time.time()
    root = Path(args.data_root) / "msd-hepatic-vessel"
    images = sorted(p for p in root.rglob("imagesTr/*.nii.gz") if not p.name.startswith("._"))
    cases = [(p, p.parent.parent / "labelsTr" / p.name) for p in images]
    cases = [c for c in cases if c[1].exists()][: args.max_units]
    vols = [load_volume(i, lab, args.size) for i, lab in cases]
    ids = [c[0].name.split(".")[0] for c in cases]
    rng, tr, val, tr_all, pool = split(len(vols), args.seed)

    def slices(idx):
        xs, ys = [], []
        for v in idx:
            xv, yv = vols[v]
            pos = np.flatnonzero(yv.reshape(len(yv), -1).max(1) > 0)
            neg = np.setdiff1d(np.arange(len(yv)), pos)
            if len(neg):
                neg = rng.choice(neg, int(np.ceil(NEG_SLICE_FRACTION * len(neg))), replace=False)
            keep = np.concatenate([pos, neg]).astype(int)
            xs.append(xv[keep])
            ys.append(yv[keep])
        return np.concatenate(xs), np.concatenate(ys)

    x_tr, y_tr = slices(tr)
    x_val, y_val = slices(val)
    print(f"msd-hepatic-vessel {args.arch} seed {args.seed}: {len(vols)} volumes -> train {len(tr)} / val {len(val)} "
          f"/ pool {len(pool)} (patient-disjoint); {len(x_tr)} training slices; load {time.time() - t0:.0f}s", flush=True)
    model, mean, std, curve, best_val = train(args.arch, x_tr, y_tr, x_val, y_val, np.ones(y_val.shape, bool),
                                              np.empty((0,)), epochs=args.epochs, seed=args.seed, batch=args.batch)
    out_dir = args.out_root / "msd-hepatic-vessel" / f"{args.arch}_seed{args.seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    probs, labs = [], []
    for v in pool:
        xv, yv = vols[v]
        p = predict(model, xv, mean, std, batch=32).astype(np.float16)
        np.savez_compressed(out_dir / f"{ids[v]}.npz", prob=p, labels=yv, valid=np.ones(yv.shape, bool))
        probs.append(p.astype(np.float32))
        labs.append(yv)
    metrics = lesion_metrics(probs, labs, [np.ones(y.shape, bool) for y in labs])
    cuts = small_cuts([vols[v][1] for v in tr_all])
    files = sorted(out_dir.glob("*.npz"))
    digest = hashlib.sha256(b"".join(hashlib.sha256(p.read_bytes()).digest() for p in files)).hexdigest()
    manifest = dict(protocol="P1 frozen model, patient-disjoint (pre-registration Amendment 3)",
                    dataset="msd-hepatic-vessel", arch=args.arch, seed=args.seed, epochs=args.epochs, size=args.size,
                    n_units=len(vols), n_train=len(tr), n_val=len(val), n_pool=len(pool),
                    train_ids=[ids[i] for i in tr], val_ids=[ids[i] for i in val], pool_ids=[ids[i] for i in pool],
                    best_val_dice=best_val, curve=curve, pool_metrics=metrics, small_cuts_from_train=cuts,
                    sha256_of_file_hashes=digest, seconds=round(time.time() - t0), env=env_info())
    (out_dir.parent / f"{args.arch}_seed{args.seed}.json").write_text(json.dumps(manifest, indent=2))
    print(f"  pool metrics {json.dumps(metrics)}\n  small cuts {cuts}\n  saved {out_dir} ({manifest['seconds']}s)",
          flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True, choices=["kvasir-seg", "fracatlas", "msd-hepatic-vessel"])
    ap.add_argument("--arch", choices=ARCHS, default="unet-r34")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--size", type=int, default=None)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--data-root", default="/tmp/pums_data")
    ap.add_argument("--smoke", action="store_true", help="<=40 units, 1 epoch, throwaway output (Amendment 3)")
    args = ap.parse_args()
    args.max_units = 40 if args.smoke else None
    args.out_root = BACKEND_ROOT / "var" / "smoke" if args.smoke else OUT_ROOT
    task08 = args.dataset == "msd-hepatic-vessel"
    args.epochs = 1 if args.smoke else (args.epochs or (25 if task08 else 60))
    if task08 and args.smoke:
        args.max_units = 6
    args.size = args.size or (256 if task08 else 352)
    (run_task08 if task08 else run_2d)(args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pums/backend/scripts/experiments/mgcrc_v2.py
"""Revision harness for lesion-level joint CRC (pre-registration Amendment 3).

Differences from mgcrc.py (the preliminary study, kept unchanged):

* Posteriors come from ONE frozen model per seed (train_frozen.py), so the
  calibration and test units are i.i.d. given the model (exact CRC setting).
* The small-lesion cut is read from the model's D_train labels (manifest), never
  from the pool.
* C1 (primary): every method calibrates on the SAME cal split; the tune split is
  used only by the lesion-equalized family choice. C2: baselines on tune U cal.
  C3: lesion-equalized with a fixed family (rho=0.5, t0=0.2) on tune U cal.
* Every quantity is computed with unit weights, so the same code serves the
  clustered bootstrap (each resampled unit keeps all its copies in one split) that
  gives CIs accounting for the finite pool and the overlapping partitions.

    python scripts/experiments/mgcrc_v2.py --dataset kvasir-seg --arch unet-r34 --seed 0
"""

from __future__ import annotations

import argparse
import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np
from scipy import ndimage
from scipy import stats as sstats

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from app.calibration.families import ScoreFamily, connectivity, default_family_grid  # noqa: E402
from app.calibration.losses import lambda_grid, unit_stats  # noqa: E402
from app.core.config import BACKEND_ROOT, REPO_ROOT  # noqa: E402

POST_ROOT = BACKEND_ROOT / "var" / "posteriors_frozen"
CACHE_ROOT = BACKEND_ROOT / "var" / "stats_v2"
OUT_ROOT = REPO_ROOT / "docs" / "research" / "results_v2"
ALPHAS = (0.05, 0.10, 0.15)
SPLIT = (0.2, 0.4, 0.4)
LTT_DELTA = 0.1
N_STRATA = 3
FIXED_FAMILY = "lesion_equalized(rho=0.5,t0=0.2)"
LOSS_BOUND = 1.0

METHODS = (
    # C1: calibrate on cal only (primary)
    "pix", "joint", "max", "dil", "strat", "ltt", "le_union", "le_max", "joint_multi",
    # C2: baselines on tune U cal
    "pix_tc", "joint_tc", "max_tc", "dil_tc", "strat_tc", "ltt_tc", "joint_multi_tc",
    # C3: fixed lesion-equalized family on tune U cal
    "le_fixed_tc",
)
METRICS = ("risk_pix", "risk_les", "risk_small", "risk_multi", "set_to_truth", "fp_to_truth", "lam")


# --------------------------------------------------------------------------- data
def lesion_equalized_conn(p, rho, t0, conn):
    """Lesion-equalized score with an explicit candidate connectivity (ablation)."""
    s = p.copy()
    lab, count = ndimage.label(p >= t0, structure=ndimage.generate_binary_structure(p.ndim, conn))
    if count == 0 or rho == 0:
        return s
    peaks = ndimage.maximum(p, lab, index=np.arange(1, count + 1))
    div = np.ones(count + 1)
    div[1:] = np.power(np.maximum(peaks, t0), rho)
    inside = lab > 0
    s[inside] = np.minimum(p[inside] / div[lab[inside]], 1.0)
    return s


def families(conn4: bool):
    fams = {f.label: f.score for f in default_family_grid()}
    fams["dilation"] = ScoreFamily("dilation").score
    if conn4:  # 2-D only: 4-connected candidates for every lesion-equalized member
        for f in default_family_grid():
            if f.kind == "lesion_equalized":
                fams["c4:" + f.label] = (lambda p, f=f: lesion_equalized_conn(p, f.rho, f.t0, 1))
    return fams


def iter_units(dataset, arch, seed):
    path = POST_ROOT / dataset / f"{arch}_seed{seed}"
    if path.is_dir():  # 3-D: one file per volume
        for f in sorted(path.glob("*.npz")):
            d = np.load(f)
            yield d["prob"].astype(np.float32), d["labels"], d["valid"]
        return
    d = np.load(path.with_suffix(".npz"))
    for p, lab, v in zip(d["prob"], d["labels"], d["valid"]):
        yield p.astype(np.float32), lab, v


def build_cache(dataset, arch, seed, tau, conn4, lambdas):
    path = CACHE_ROOT / f"{dataset}_{arch}_seed{seed}_tau{tau:g}{'_c4' if conn4 else ''}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    fams = families(conn4)
    stats = {k: [] for k in fams}
    mass, lesions = [], []  # lesions: per-unit arrays for the error analysis
    t = time.time()
    for p, lab, v in iter_units(dataset, arch, seed):
        p = np.where(v, p, 0.0).astype(np.float32)
        mass.append(float(p.sum()))
        for name, score in fams.items():
            s = score(p)
            s[~v] = 0.0
            stats[name].append(unit_stats(s, lab, tau=tau, lambdas=lambdas))
        bg = p[(lab == 0) & v]
        k = int(lab.max())
        peaks = ndimage.maximum(p, lab, index=np.arange(1, k + 1)) if k else np.zeros(0)
        sizes = np.bincount(lab.ravel(), minlength=k + 1)[1:] if k else np.zeros(0, int)
        lesions.append(dict(peak=np.asarray(peaks, float), size=np.asarray(sizes),
                            bg_q99=float(np.percentile(bg, 99)) if bg.size else 0.0))
    out = dict(stats=stats, mass=np.asarray(mass), lesions=lesions)
    print(f"    stats: {len(mass)} units x {len(fams)} families ({time.time() - t:.0f}s)", flush=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(pickle.dumps(out))
    return out


class Mats:
    """Per-unit loss curves of one family on the grid (rows = units with GT)."""

    def __init__(self, units, small_cut, lambdas):
        cut = 1.0 - lambdas
        n, g = len(units), lambdas.size
        self.pix = np.empty((n, g), np.float32)
        self.les = np.empty((n, g), np.float32)
        self.small = np.zeros((n, g), np.float32)
        self.has_small = np.zeros(n, bool)
        self.size = np.empty((n, g), np.float64)
        self.n_gt = np.empty(n, np.float64)
        self.multi = np.zeros(n, bool)
        for i, u in enumerate(units):
            self.pix[i] = u.pix_miss / u.n_gt
            st = np.sort(u.lesion_stat)
            self.les[i] = np.searchsorted(st, cut, side="left") / st.size
            sm = np.sort(u.lesion_stat[u.lesion_size < small_cut])
            if sm.size:
                self.has_small[i] = True
                self.small[i] = np.searchsorted(sm, cut, side="left") / sm.size
            self.size[i] = u.set_size
            self.n_gt[i] = u.n_gt
            self.multi[i] = u.k >= 2
        self.mx = np.maximum(np.maximum(self.pix, self.les), np.where(self.has_small[:, None], self.small, 0.0))


# --------------------------------------------------------------------------- rules (weighted)
def curves(m: Mats, idx, w):
    """Weighted mean loss curves and the effective n of each risk on subset idx."""
    wi = w[idx]
    out = {}
    for name, rows in (("pix", None), ("les", None), ("small", m.has_small[idx]), ("multi", m.multi[idx])):
        ww = wi if rows is None else wi * rows
        n = ww.sum()
        mat = {"pix": m.pix, "les": m.les, "small": m.small, "multi": m.les}[name]
        out[name] = ((ww @ mat[idx]) / n if n > 0 else np.zeros(mat.shape[1]), n)
    out["max"] = None
    return out


def max_curve(m: Mats, idx, w):
    wi = w[idx]
    return (wi @ m.mx[idx]) / wi.sum(), wi.sum()


def crc_idx(curve, n, alpha):
    if n <= 0:
        return 0
    feas = np.flatnonzero((n * curve + LOSS_BOUND) / (n + 1) <= alpha)
    return int(feas[0]) if feas.size else curve.size - 1


def union_idx(cv, alpha, risks=("pix", "les", "small")):
    return max(crc_idx(*cv[r], alpha) for r in risks)


def hb_p(r_hat, n, alpha):
    r = np.minimum(r_hat, alpha)
    with np.errstate(divide="ignore", invalid="ignore"):
        h1 = r * np.log(r / alpha) + (1 - r) * np.log((1 - r) / (1 - alpha))
    h1 = np.where(r <= 0, -np.log(1 - alpha), h1)
    return np.minimum(np.minimum(np.exp(-n * h1), np.e * sstats.binom.cdf(np.ceil(n * r_hat), int(round(n)), alpha)), 1)


def ltt_idx(cv, alpha, delta=LTT_DELTA):
    p = np.maximum.reduce([hb_p(cv[r][0], cv[r][1], alpha) for r in ("pix", "les", "small") if cv[r][1] > 0])
    i = p.size - 1
    while i - 1 >= 0 and p[i - 1] <= delta:
        i -= 1
    return i


def evaluate(m: Mats, idx, w, index):
    wi = w[idx]
    ii = np.broadcast_to(np.asarray(index), (idx.size,))
    rows = idx
    pix = m.pix[rows, ii]
    les = m.les[rows, ii]
    size = m.size[rows, ii]
    hs, mu = m.has_small[rows], m.multi[rows]
    gt = (wi * m.n_gt[rows]).sum()

    def wmean(v, mask=None):
        ww = wi if mask is None else wi * mask
        return float((ww * v).sum() / ww.sum()) if ww.sum() > 0 else np.nan

    covered = m.n_gt[rows] * (1 - pix)
    return (wmean(pix), wmean(les), wmean(m.small[rows, ii], hs), wmean(les, mu),
            float((wi * size).sum() / gt), float((wi * (size - covered)).sum() / gt), float(np.mean(ii)))


# --------------------------------------------------------------------------- one partition
def run_partition(fam_mats, base, dil, le_labels, tune, cal, test, w, mass, alphas, lambdas, fixed):
    tc = np.concatenate([tune, cal])
    res = np.full((len(alphas), len(METHODS), len(METRICS)), np.nan)
    chosen = []
    cv = {"cal": curves(base, cal, w), "tc": curves(base, tc, w)}
    cvd = {"cal": curves(dil, cal, w), "tc": curves(dil, tc, w)}
    mx = {"cal": max_curve(base, cal, w), "tc": max_curve(base, tc, w)}
    tune_cv = {lab: curves(fam_mats[lab], tune, w) for lab in le_labels}
    tune_mx = {lab: max_curve(fam_mats[lab], tune, w) for lab in le_labels}
    gt_tune = {lab: (w[tune] * fam_mats[lab].n_gt[tune]).sum() for lab in le_labels}
    fixed_cv = curves(fam_mats[fixed], tc, w)
    strata = {}
    for sub in ("cal", "tc"):
        cidx = cal if sub == "cal" else tc
        q = np.quantile(mass[cidx], np.linspace(0, 1, N_STRATA + 1)[1:-1])
        sc, st = np.digitize(mass[cidx], q), np.digitize(mass[test], q)
        strata[sub] = (st, {g: curves(base, cidx[sc == g], w) for g in range(N_STRATA)})
    for ai, a in enumerate(alphas):
        out = {}
        for sfx, sub in (("", "cal"), ("_tc", "tc")):
            c = cv[sub]
            out["pix" + sfx] = (base, crc_idx(*c["pix"], a))
            out["joint" + sfx] = (base, union_idx(c, a))
            out["max" + sfx] = (base, crc_idx(*mx[sub], a))
            out["dil" + sfx] = (dil, union_idx(cvd[sub], a))
            out["ltt" + sfx] = (base, ltt_idx(c, a))
            out["joint_multi" + sfx] = (base, union_idx(c, a, ("pix", "les", "small", "multi")) if c["multi"][1] > 0
                                        else union_idx(c, a))
            # CCRA-S-style strata of total predicted mass; cutpoints from the calibration units
            st, scv = strata[sub]
            per = {g: union_idx(scv[g], a) for g in range(N_STRATA)}
            out["strat" + sfx] = (base, np.asarray([per[g] for g in st]))
        # ours: family chosen on tune (smallest tuned set), calibrated on cal
        for rule in ("union", "max"):
            best, best_size, best_i = None, np.inf, 0
            for lab in le_labels:
                m = fam_mats[lab]
                i = union_idx(tune_cv[lab], a) if rule == "union" else crc_idx(*tune_mx[lab], a)
                size = (w[tune] * m.size[tune, i]).sum() / gt_tune[lab]
                if size < best_size - 1e-12:
                    best, best_size = lab, size
            m = fam_mats[best]
            i = union_idx(curves(m, cal, w), a) if rule == "union" else crc_idx(*max_curve(m, cal, w), a)
            out[f"le_{rule}"] = (m, i)
            if rule == "union":
                chosen.append(best)
        out["le_fixed_tc"] = (fam_mats[fixed], union_idx(fixed_cv, a))
        for mi, name in enumerate(METHODS):
            m, i = out[name]
            res[ai, mi] = evaluate(m, test, w, i)
    return res, chosen


def partitions(n, rng, reps, units=None):
    """Random tune/cal/test partitions over distinct units (clusters)."""
    n_tune, n_cal = int(SPLIT[0] * n), int(SPLIT[1] * n)
    for _ in range(reps):
        perm = rng.permutation(n)
        yield perm[:n_tune], perm[n_tune:n_tune + n_cal], perm[n_tune + n_cal:]


def run(dataset, arch, seed, reps, boot, boot_reps, tau, small_q, conn4, ncal, alphas, workers_seed=0):
    lambdas = lambda_grid()
    manifest = json.loads((POST_ROOT / dataset / f"{arch}_seed{seed}.json").read_text())
    small_cut = manifest["small_cuts_from_train"][f"q{small_q:g}"]
    cache = build_cache(dataset, arch, seed, tau, conn4, lambdas)
    keep = [i for i, u in enumerate(cache["stats"]["threshold"]) if u.n_gt > 0 and u.k > 0]
    mass = cache["mass"][keep]
    fam_mats = {k: Mats([v[i] for i in keep], small_cut, lambdas) for k, v in cache["stats"].items()}
    base, dil = fam_mats["threshold"], fam_mats["dilation"]
    le_labels = [f.label for f in default_family_grid()]
    if conn4:
        le_labels = ["threshold"] + [k for k in fam_mats if k.startswith("c4:")]
    n = len(keep)
    rng = np.random.default_rng(seed * 7919 + 17)
    w1 = np.ones(n)

    def one(tune, cal, test, w):
        if ncal:
            cal = cal[:ncal]
        return run_partition(fam_mats, base, dil, le_labels, tune, cal, test, w, mass, alphas, lambdas, FIXED_FAMILY)

    t = time.time()
    main_res, chosen = [], []
    for tune, cal, test in partitions(n, rng, reps):
        r, c = one(tune, cal, test, w1)
        main_res.append(r)
        chosen.extend(c)
    main_res = np.stack(main_res)  # [R, A, M, K]
    print(f"    {reps} partitions: {time.time() - t:.0f}s", flush=True)

    boot_res = boot_rel = None
    if boot:
        t = time.time()
        boot_res, boot_rel = [], []
        j, k_str = METHODS.index("joint"), METRICS.index("set_to_truth")
        for b in range(boot):
            draw = rng.integers(0, n, n)
            ids, counts = np.unique(draw, return_counts=True)
            w = np.zeros(n)
            w[ids] = counts
            acc = []
            for tune, cal, test in partitions(ids.size, rng, boot_reps):
                acc.append(one(ids[tune], ids[cal], ids[test], w)[0])
            acc = np.stack(acc)
            boot_res.append(np.nanmean(acc, axis=0))  # mean over partitions
            boot_rel.append(np.nanmean(acc[:, :, :, k_str] / acc[:, :, j:j + 1, k_str] - 1, axis=0))
        boot_res, boot_rel = np.stack(boot_res), np.stack(boot_rel)  # [B, A, M, K], [B, A, M]
        print(f"    bootstrap {boot} x {boot_reps}: {time.time() - t:.0f}s", flush=True)

    lesions = [cache["lesions"][i] for i in keep]
    meta = dict(dataset=dataset, arch=arch, seed=seed, tau=tau, small_q=small_q, small_cut=small_cut, conn4=conn4,
                ncal=ncal, reps=reps, boot=boot, boot_reps=boot_reps, n_units=n, n_multi=int(base.multi.sum()),
                n_small=int(base.has_small.sum()), split=SPLIT, ltt_delta=LTT_DELTA, grid=int(lambdas.size),
                alphas=list(alphas), methods=list(METHODS), metrics=list(METRICS),
                n_cal=int(SPLIT[1] * n) if not ncal else min(ncal, int(SPLIT[1] * n)), n_tune=int(SPLIT[0] * n),
                pool_metrics=manifest["pool_metrics"], fixed_family=FIXED_FAMILY)
    return dict(meta=meta, main=main_res, boot=boot_res, boot_rel=boot_rel, chosen=chosen, lesions=lesions,
                base_stat_les=[u.lesion_stat for u in [cache["stats"]["threshold"][i] for i in keep]])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--arch", default="unet-r34")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--reps", type=int, default=1000)
    ap.add_argument("--boot", type=int, default=200)
    ap.add_argument("--boot-reps", type=int, default=100)
    ap.add_argument("--tau", type=float, default=0.5)
    ap.add_argument("--small-q", type=float, default=33.0)
    ap.add_argument("--conn4", action="store_true")
    ap.add_argument("--ncal", type=int, default=0)
    ap.add_argument("--tag", default="main")
    args = ap.parse_args()
    t = time.time()
    out = run(args.dataset, args.arch, args.seed, args.reps, args.boot, args.boot_reps, args.tau,
              args.small_q, args.conn4, args.ncal, ALPHAS)
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    stem = f"{args.tag}_{args.dataset}_{args.arch}_seed{args.seed}"
    (OUT_ROOT / f"{stem}.pkl").write_bytes(pickle.dumps(out))
    m = out["main"].mean(0)
    print(f"{stem}: {time.time() - t:.0f}s  n={out['meta']['n_units']} multi={out['meta']['n_multi']}")
    for ai, a in enumerate(ALPHAS):
        for mi, name in enumerate(METHODS):
            v = m[ai, mi]
            print(f"  a={a:.2f} {name:<15} pix {v[0]:.3f} les {v[1]:.3f} small {v[2]:.3f} multi {v[3]:.3f} "
                  f"STR {v[4]:.3f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pums/backend/scripts/experiments/aggregate_v2.py
"""Aggregate the revision runs (Amendment 3) into one summary JSON + tables for the paper.

Seed handling: model seeds are independent replicates. Point estimates are the
mean over seeds of each seed's partition mean. CIs come from the clustered
bootstrap: replicate b of the seed-averaged estimand is the mean over seeds of
each seed's b-th bootstrap replicate (independent resampling per seed's pool).

    python scripts/experiments/aggregate_v2.py
"""

from __future__ import annotations

import collections
import json
import pickle
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
sys.path.insert(0, str(Path(__file__).resolve().parent))

import mgcrc_v2 as V  # noqa: E402

from app.core.config import REPO_ROOT  # noqa: E402

RES = REPO_ROOT / "docs" / "research" / "results_v2"
SLACK = 0.005
M, K = list(V.METHODS), list(V.METRICS)


def load(tag):
    runs = collections.defaultdict(dict)
    for f in sorted(RES.glob(f"{tag}_*.pkl")):
        d = pickle.loads(f.read_bytes())
        meta = d["meta"]
        runs[(meta["dataset"], meta["arch"])][meta["seed"]] = d
    return runs


def ci(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return [float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))] if x.size else [np.nan, np.nan]


def seed_avg_boot(seeds, getter):
    """Replicate-wise mean over seeds of getter(run)[b] -> array over b."""
    arrs = [getter(r) for r in seeds.values() if r.get("boot") is not None]
    if not arrs:
        return None
    b = min(len(a) for a in arrs)
    return np.nanmean(np.stack([a[:b] for a in arrs]), axis=0)


def summarise_config(seeds: dict):
    any_run = next(iter(seeds.values()))
    alphas = any_run["meta"]["alphas"]
    out = dict(seeds=sorted(seeds), alphas=alphas)
    # pool and backbone facts
    pm = [r["meta"]["pool_metrics"] for r in seeds.values()]
    out["pool_metrics"] = {k: dict(mean=float(np.mean([p[k] for p in pm])), sd=float(np.std([p[k] for p in pm], ddof=1))
                                   if len(pm) > 1 else 0.0, per_seed=[p[k] for p in pm])
                           for k in ("dice", "iou", "lesion_recall", "lesion_precision", "fp_components_per_unit")}
    out["counts"] = {k: [r["meta"][k] for r in seeds.values()] for k in ("n_units", "n_multi", "n_small", "n_cal", "n_tune")}
    out["small_cut"] = [r["meta"]["small_cut"] for r in seeds.values()]
    per = {}
    for ai, a in enumerate(alphas):
        for mi, m in enumerate(M):
            row = {}
            for ki, k in enumerate(K):
                point = float(np.mean([np.nanmean(r["main"][:, ai, mi, ki]) for r in seeds.values()]))
                split = ci(np.concatenate([r["main"][:, ai, mi, ki] for r in seeds.values()]))
                bt = seed_avg_boot(seeds, lambda r: r["boot"][:, ai, mi, ki])
                row[k] = dict(mean=point, split=split, ci=ci(bt) if bt is not None else None,
                              per_seed=[float(np.nanmean(r["main"][:, ai, mi, ki])) for r in seeds.values()])
            # paired relative change in set size vs joint thresholding (C1 reference)
            j, s = M.index("joint"), K.index("set_to_truth")
            rel = np.concatenate([r["main"][:, ai, mi, s] / r["main"][:, ai, j, s] - 1 for r in seeds.values()])
            bt = seed_avg_boot(seeds, lambda r: r["boot_rel"][:, ai, mi])
            row["rel_vs_joint"] = dict(mean=float(np.nanmean(rel)), split=ci(rel), p_smaller=float(np.mean(rel < 0)),
                                       ci=ci(bt) if bt is not None else None,
                                       per_seed=[float(np.nanmean(r["main"][:, ai, mi, s] / r["main"][:, ai, j, s] - 1))
                                                 for r in seeds.values()])
            claims = {"pix": ["risk_pix"], "pix_tc": ["risk_pix"]}.get(m, ["risk_pix", "risk_les", "risk_small"])
            if m.startswith("joint_multi"):
                claims = ["risk_pix", "risk_les", "risk_small", "risk_multi"]
            if m.startswith("max"):
                claims = ["risk_pix", "risk_les"]  # Proposition 1: small-lesion risk only at alpha/pi
            if m == "le_max":
                claims = ["risk_pix", "risk_les"]
            row["claims"] = claims
            row["valid"] = bool(all(row[c]["mean"] <= a + SLACK for c in claims if np.isfinite(row[c]["mean"])))
            per[f"{a:.2f}|{m}"] = row
    out["methods"] = per
    # gap (RQ1): at the pixel-CRC threshold calibrated on cal (C1)
    gap = {}
    for ai, a in enumerate(alphas):
        mi = M.index("pix")
        g = {}
        for name, k in (("G_all", "risk_les"), ("G_multi", "risk_multi"), ("G_small", "risk_small")):
            ki = K.index(k)
            point = float(np.mean([np.nanmean(r["main"][:, ai, mi, ki]) for r in seeds.values()])) / a
            bt = seed_avg_boot(seeds, lambda r: r["boot"][:, ai, mi, ki])
            g[name] = dict(mean=point, ci=[c / a for c in ci(bt)] if bt is not None else None,
                           split=[c / a for c in ci(np.concatenate([r["main"][:, ai, mi, ki] for r in seeds.values()]))])
        gap[f"{a:.2f}"] = g
    out["gap"] = gap
    g10 = gap["0.10"]
    out["H1"] = ("PROCEED" if max(g10["G_multi"]["mean"], g10["G_small"]["mean"]) >= 1.5 else
                 "KILL" if all(max(gap[f"{a:.2f}"]["G_multi"]["mean"], gap[f"{a:.2f}"]["G_small"]["mean"]) <= 1.1
                               for a in alphas) else "INCONCLUSIVE")
    # H2 per alpha (C1)
    h2 = {}
    for a in alphas:
        r = per[f"{a:.2f}|le_union"]
        rv = r["rel_vs_joint"]
        ok_ci = rv["ci"] is not None and rv["ci"][1] < 0
        h2[f"{a:.2f}"] = dict(rel=rv["mean"], ci=rv["ci"], p_smaller=rv["p_smaller"], valid=r["valid"],
                             supported=bool(rv["mean"] < 0 and rv["p_smaller"] >= 0.95 and ok_ci and r["valid"]),
                             failed=bool(rv["mean"] >= 0))
    out["H2"] = h2
    # H3 (joint + multi-lesion risk), only if >= 30 multi-lesion pool units
    n_multi = min(out["counts"]["n_multi"])
    h3 = {}
    for a in alphas:
        jm, j = per[f"{a:.2f}|joint_multi"], per[f"{a:.2f}|joint"]
        h3[f"{a:.2f}"] = dict(multi_joint=j["risk_multi"]["mean"], multi_joint_ci=j["risk_multi"]["ci"],
                              multi_with=jm["risk_multi"]["mean"], multi_with_ci=jm["risk_multi"]["ci"],
                              set_change=jm["rel_vs_joint"]["mean"], set_change_ci=jm["rel_vs_joint"]["ci"],
                              bg_joint=j["fp_to_truth"]["mean"], bg_with=jm["fp_to_truth"]["mean"],
                              supported=bool(jm["risk_multi"]["mean"] <= a + SLACK) if n_multi >= 30 else None)
    out["H3"] = dict(evaluable=n_multi >= 30, n_multi_pool=out["counts"]["n_multi"], per_alpha=h3)
    chosen = collections.Counter()
    for r in seeds.values():
        chosen.update(r["chosen"])
    total = sum(chosen.values())
    out["chosen_family"] = {k: v / total for k, v in chosen.most_common()}
    return out


def detectability(seeds: dict):
    """Per small / all lesion: peak posterior vs the unit's 99th-percentile background."""
    rows = []
    for r in seeds.values():
        cut = r["meta"]["small_cut"]
        for les in r["lesions"]:
            for pk, sz in zip(les["peak"], les["size"]):
                rows.append((pk, sz, les["bg_q99"], sz < cut))
    a = np.asarray(rows, float) if rows else np.zeros((0, 4))
    small = a[a[:, 3] == 1] if len(a) else a
    def frac(x, t0):
        return float(np.mean(x[:, 0] > np.maximum(x[:, 2], t0))) if len(x) else float("nan")
    return dict(n_lesions=int(len(a)), n_small=int(len(small)),
                small_peak_above_bg=frac(small, 0.0), small_peak_above_bg_and_t0_0p2=frac(small, 0.2),
                all_peak_above_bg=frac(a, 0.0), small_peak_median=float(np.median(small[:, 0])) if len(small) else None,
                small_bg_q99_median=float(np.median(small[:, 2])) if len(small) else None)


def missed_lesion_profile(seeds: dict, alpha=0.10, reps=200):
    """Lesions missed by joint thresholding (C1) on test units: size and peak vs bg, over random partitions."""
    lambdas = V.lambda_grid()
    missed, found = [], []
    for seed, r in seeds.items():
        meta = r["meta"]
        cache = V.build_cache(meta["dataset"], meta["arch"], seed, meta["tau"], False, lambdas)
        keep = [i for i, u in enumerate(cache["stats"]["threshold"]) if u.n_gt > 0 and u.k > 0]
        units = [cache["stats"]["threshold"][i] for i in keep]
        base = V.Mats(units, meta["small_cut"], lambdas)
        les = [cache["lesions"][i] for i in keep]
        rng = np.random.default_rng(99 + seed)
        n = len(keep)
        w = np.ones(n)
        for tune, cal, test in V.partitions(n, rng, reps):
            idx = V.union_idx(V.curves(base, cal, w), alpha)
            cutoff = 1 - lambdas[idx]
            for t in test:
                u, l = units[t], les[t]
                for k in range(u.k):
                    rec = (float(u.lesion_size[k]), float(l["peak"][k]), float(l["bg_q99"]),
                           bool(u.lesion_size[k] < meta["small_cut"]))
                    (missed if u.lesion_stat[k] < cutoff else found).append(rec)
    m, f = np.asarray(missed, float), np.asarray(found, float)
    def desc(x):
        if not len(x):
            return None
        return dict(n=int(len(x)), median_size=float(np.median(x[:, 0])), frac_small=float(np.mean(x[:, 3])),
                    median_peak=float(np.median(x[:, 1])), frac_peak_below_bg=float(np.mean(x[:, 1] <= x[:, 2])),
                    frac_no_candidate_t0_0p2=float(np.mean(x[:, 1] < 0.2)))
    return dict(alpha=alpha, missed=desc(m), found=desc(f))


def main():
    summary = {}
    main_runs = load("main")
    for (ds, arch), seeds in sorted(main_runs.items()):
        key = f"{ds}|{arch}"
        s = summarise_config(seeds)
        s["detectability"] = detectability(seeds)
        try:
            s["missed_profile"] = missed_lesion_profile(seeds)
        except FileNotFoundError:
            s["missed_profile"] = None
        summary[key] = s
        print(f"{key}: seeds {s['seeds']}  H1 {s['H1']}  H2 " +
              " ".join(f"{a}:{'SUPPORTED' if v['supported'] else 'FAILED' if v['failed'] else 'partial'}"
                       f"({100 * v['rel']:+.1f}%)" for a, v in s["H2"].items()))
    # ablations: any tag other than main
    abl = {}
    for f in sorted(RES.glob("*.pkl")):
        tag = f.name.split("_")[0]
        if tag == "main":
            continue
        d = pickle.loads(f.read_bytes())
        meta = d["meta"]
        key = f"{tag}|{meta['dataset']}|{meta['arch']}"
        abl.setdefault(key, {})[meta["seed"]] = d
    abl_out = {}
    for key, seeds in abl.items():
        s = summarise_config(seeds)
        abl_out[key] = dict(meta={k: next(iter(seeds.values()))["meta"][k] for k in ("tau", "small_q", "conn4", "ncal")},
                            gap=s["gap"], H2=s["H2"],
                            le_rel={a: s["methods"][f"{a:.2f}|le_union"]["rel_vs_joint"] for a in s["alphas"]},
                            joint_vs_pix={a: s["methods"][f"{a:.2f}|pix"]["rel_vs_joint"] for a in s["alphas"]},
                            validity={m: s["methods"][f"0.10|{m}"]["valid"] for m in M})
    summary["_ablations"] = abl_out
    (RES / "summary_v2.json").write_text(json.dumps(summary, indent=1, default=float))
    print(f"wrote {RES / 'summary_v2.json'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pums/backend/app/core/config.py
"""Kaggle paths: large intermediates in /tmp, results in /kaggle/working (kept with the version)."""
from pathlib import Path

BACKEND_ROOT = Path("/tmp/pums_backend")      # posteriors, stats caches (not saved)
REPO_ROOT = Path("/kaggle/working/pums")      # docs/research/results_v2 is saved with the notebook version

## Data (official sources)

In [ ]:
DATA = "/tmp/pums_data"
os.makedirs(DATA, exist_ok=True)
import hashlib
# Integrity is checked against SHA-256 / size of the copies used in the preliminary study, so falling
# back to --no-check-certificate (Simula's TLS certificate fails verification) cannot admit a wrong file.
EXPECTED = {"kvasir-seg.zip": ("03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7", 46227172),
            "fracatlas.zip": ("b67ec2d290a022b3dcf47f78e9a37f7edcc80592c0571f439355bf00bd9f0e23", 338436460),
            "task08.tar": (None, 9347254784)}
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()
def fetch(url, dest):
    if not os.path.exists(dest):
        base = ["wget", "-q", "--tries=20", "--continue", "-O", dest, url]
        if subprocess.run(base).returncode != 0:
            print("certificate/transfer problem; retrying without TLS verification (checksum enforced below)")
            if os.path.exists(dest):
                os.remove(dest)
            subprocess.run(base[:1] + ["--no-check-certificate"] + base[1:], check=True)
    digest, size = EXPECTED[os.path.basename(dest)]
    got = os.path.getsize(dest)
    assert got == size, f"{dest}: size {got} != expected {size}"
    if digest:
        assert sha256(dest) == digest, f"{dest}: SHA-256 mismatch"
    print(dest, got // 2**20, "MB, verified")
def unpack(archive, target):
    if not os.path.isdir(target):
        os.makedirs(target)
        shutil.unpack_archive(archive, target)
if JOB == "2d":
    fetch("https://datasets.simula.no/downloads/kvasir-seg.zip", f"{DATA}/kvasir-seg.zip")
    unpack(f"{DATA}/kvasir-seg.zip", f"{DATA}/kvasir-seg")
    fetch("https://ndownloader.figshare.com/files/65518038", f"{DATA}/fracatlas.zip")
    unpack(f"{DATA}/fracatlas.zip", f"{DATA}/fracatlas")
else:
    fetch("https://msd-for-monai.s3-us-west-2.amazonaws.com/Task08_HepaticVessel.tar", f"{DATA}/task08.tar")
    unpack(f"{DATA}/task08.tar", f"{DATA}/msd-hepatic-vessel")
    os.remove(f"{DATA}/task08.tar")
os.system(f"du -sh {DATA}/* | cat")

## Smoke test (Amendment 3: <=40 units, 1 epoch; outputs deleted)

In [ ]:
EXP = "/kaggle/working/pums/backend/scripts/experiments"
DATASETS = ["kvasir-seg", "fracatlas"] if JOB == "2d" else ["msd-hepatic-vessel"]
if RUN_SMOKE:
    for ds in DATASETS:
        for a in ARCHS:
            r = subprocess.run([sys.executable, f"{EXP}/kaggle_train.py", "--dataset", ds, "--arch", a, "--seed", "0",
                                "--smoke", "--data-root", DATA], capture_output=True, text=True)
            print(ds, a, "OK" if r.returncode == 0 else "FAILED"); print(r.stdout[-1500:], r.stderr[-3000:])
            assert r.returncode == 0, "smoke test failed: stop and report the error above"
    shutil.rmtree("/tmp/pums_backend/var/smoke", ignore_errors=True)

## Train the frozen backbones

In [ ]:
LOGS = "/kaggle/working/pums/docs/research/results_v2/logs"
os.makedirs(LOGS, exist_ok=True)
def sh(cmd, log):
    with open(log, "w") as f:
        p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    print(open(log).read()[-800:]); assert p.returncode == 0, f"failed: {log}"
for ds in DATASETS:
    for a in ARCHS:
        for s in SEEDS:
            t = time.time()
            sh([sys.executable, f"{EXP}/kaggle_train.py", "--dataset", ds, "--arch", a, "--seed", str(s), "--data-root", DATA],
               f"{LOGS}/train_{ds}_{a}_{s}.log")
            print(f"== {ds} {a} seed {s}: {time.time()-t:.0f}s", flush=True)

## Conformal calibration, bootstrap, ablations

In [ ]:
for ds in DATASETS:
    for a in ARCHS:
        for s in SEEDS:
            sh([sys.executable, f"{EXP}/mgcrc_v2.py", "--dataset", ds, "--arch", a, "--seed", str(s), "--tag", "main"],
               f"{LOGS}/main_{ds}_{a}_{s}.log")
ABL = [("abltau25", ["--tau", "0.25"]), ("abltau75", ["--tau", "0.75"]), ("ablq25", ["--small-q", "25"]),
       ("ablq50", ["--small-q", "50"]), ("ablncal50", ["--ncal", "50"]), ("ablncal100", ["--ncal", "100"]),
       ("ablncal200", ["--ncal", "200"])]
if JOB == "2d":
    ABL.append(("ablconn4", ["--conn4"]))
if "unet-r34" in ARCHS:
    for ds in DATASETS:
        for s in SEEDS:
            for tag, extra in ABL:
                sh([sys.executable, f"{EXP}/mgcrc_v2.py", "--dataset", ds, "--arch", "unet-r34", "--seed", str(s),
                    "--tag", tag, "--boot", "50", "--boot-reps", "50", *extra], f"{LOGS}/{tag}_{ds}_{s}.log")

## Aggregate and package

In [ ]:
sh([sys.executable, f"{EXP}/aggregate_v2.py"], f"{LOGS}/aggregate_{JOB}.log")
man = "/kaggle/working/pums/docs/research/results_v2/manifests"
os.makedirs(man, exist_ok=True)
for root, _, files in os.walk("/tmp/pums_backend/var/posteriors_frozen"):
    for f in files:
        if f.endswith(".json"):
            shutil.copy(os.path.join(root, f), f"{man}/{os.path.basename(root)}__{f}")
out = shutil.make_archive(f"/kaggle/working/revision_results_{JOB}_{'-'.join(ARCHS)}", "zip",
                          "/kaggle/working/pums/docs/research", "results_v2")
print(out, os.path.getsize(out) // 2**20, "MB")

## After the run
Open the version's **Output** tab and download `revision_results_*.zip`. Put every zip (one per session) into
`docs/research/results_v2_kaggle/` in the repository and tell Claude. It merges them and runs the final aggregation,
figures, and paper build locally.